In [1]:
from pathlib import Path
import json
import csv
import torch

# Rechercher le checkpoint dans les Inputs Kaggle
matches = [
    path
    for path in Path("/kaggle/input").rglob("best_model.pt")
    if path.parent.name == "hazelnut-baseline-001"
]

assert len(matches) == 1, (
    f"Checkpoint introuvable ou ambigu : {matches}"
)

RUN_DIR = matches[0].parent

# Lire les artefacts persistés
with (RUN_DIR / "run_info.json").open(encoding="utf-8") as f:
    run_info = json.load(f)

with (RUN_DIR / "history.csv").open(
    encoding="utf-8", newline=""
) as f:
    history = list(csv.DictReader(f))

checkpoint = torch.load(
    RUN_DIR / "best_model.pt",
    map_location="cpu",
    weights_only=True,
)

assert len(history) == 150
assert checkpoint["epoch"] == 147

print("Saved outputs:", RUN_DIR)
print("Checkpoint epoch:", checkpoint["epoch"])
print("Best validation MSE:", checkpoint["best_val_mse"])
print("Training commit:", checkpoint["git_commit"])
print("Persistent checkpoint: OK")


Saved outputs: /kaggle/input/notebooks/marwanedrissi/industrialdefect-ai-cae-hazelnut-training/cae/hazelnut/hazelnut-baseline-001
Checkpoint epoch: 147
Best validation MSE: 0.00020595180364743544
Training commit: 742b8346e664b4efc50410b3707bdb2db6908c1f
Persistent checkpoint: OK


In [2]:
import subprocess
import sys

REPO_URL = "https://github.com/MRdrissi/industrial-defect-ai.git"
REPO_DIR = Path("/kaggle/working/industrial-defect-ai-restore")

assert not REPO_DIR.exists(), "Utiliser un répertoire neuf."

subprocess.run(
    ["git", "clone", REPO_URL, str(REPO_DIR)],
    check=True,
)

subprocess.run(
    [
        "git", "-C", str(REPO_DIR),
        "checkout", "--detach", checkpoint["git_commit"],
    ],
    check=True,
)

sys.path.insert(0, str(REPO_DIR))

from src.models.convolutional_autoencoder import (
    CAEConfig,
    ConvolutionalAutoencoder,
)

params = checkpoint["model_config"]

config = CAEConfig(
    input_channels=params["input_channels"],
    image_size=tuple(params["image_size"]),
    encoder_channels=tuple(params["encoder_channels"]),
    latent_channels=params["latent_channels"],
    seed=params["seed"],
)

# Nouveau modèle, sans entraînement
restored_model = ConvolutionalAutoencoder(config).cpu()

restored_model.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True,
)

restored_model.eval()

print("Model restored successfully")
print("Device:", next(restored_model.parameters()).device)
print("Architecture:", config)


Cloning into '/kaggle/working/industrial-defect-ai-restore'...


Model restored successfully
Device: cpu
Architecture: CAEConfig(input_channels=3, image_size=(256, 256), encoder_channels=(32, 64, 128), latent_channels=128, seed=42)


HEAD is now at 742b834 feat(US-03-04): add Hazelnut Kaggle preflight notebook


In [3]:
import hashlib
import torch.nn.functional as F

from src.data.mvtec_dataset import MVTecDataset
from src.data.transforms import MVTecTransform
from src.data.dataloaders import create_dataloader

manifest_path = (
    REPO_DIR
    / run_info["experiment"]["dataset"]["manifest"]
)

actual_sha = hashlib.sha256(
    manifest_path.read_bytes()
).hexdigest()

assert actual_sha == checkpoint["manifest_sha256"]

good_dirs = list(
    Path("/kaggle/input").rglob("hazelnut/train/good")
)

assert len(good_dirs) == 1
dataset_root = good_dirs[0].parents[2]

transform = MVTecTransform(
    size=config.image_size,
    profile="cae",
)

val_dataset = MVTecDataset(
    dataset_root=dataset_root,
    manifest_path=manifest_path,
    category="hazelnut",
    split="val",
    transform=transform,
)

val_loader = create_dataloader(
    val_dataset,
    batch_size=16,
    num_workers=0,
    seed=config.seed,
)

assert len(val_dataset) == 78

total_squared_error = 0.0
total_elements = 0

with torch.inference_mode():
    for batch in val_loader:
        images = batch["image"].cpu()

        reconstructions = restored_model(images)

        assert reconstructions.shape == images.shape
        assert torch.isfinite(reconstructions).all().item()

        total_squared_error += F.mse_loss(
            reconstructions,
            images,
            reduction="sum",
        ).item()

        total_elements += images.numel()

restored_mse = total_squared_error / total_elements

expected_mse = checkpoint["best_val_mse"]
absolute_difference = abs(restored_mse - expected_mse)
relative_difference = absolute_difference / expected_mse

print(f"Original validation MSE: {expected_mse:.10f}")
print(f"Restored validation MSE: {restored_mse:.10f}")
print(f"Absolute difference: {absolute_difference:.10f}")
print(f"Relative difference: {relative_difference:.6%}")

tolerance = max(1e-7, expected_mse * 1e-3)
assert absolute_difference <= tolerance

print("Independent CPU restoration: OK")


Original validation MSE: 0.0002059518
Restored validation MSE: 0.0002059518
Absolute difference: 0.0000000000
Relative difference: 0.000009%
Independent CPU restoration: OK


In [4]:

import json
import hashlib
import platform
from pathlib import Path

REPORT_DIR = Path(
    "/kaggle/working/cae/hazelnut/restore-check-001"
)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

checkpoint_path = RUN_DIR / "best_model.pt"

report = {
    "category": "hazelnut",
    "run_id": "hazelnut-baseline-001",
    "status": "passed",
    "device": "cpu",

    "training_commit": checkpoint["git_commit"],
    "manifest_sha256": checkpoint["manifest_sha256"],

    "checkpoint": {
        "source_path": str(checkpoint_path),
        "sha256": hashlib.sha256(
            checkpoint_path.read_bytes()
        ).hexdigest(),
        "epoch": checkpoint["epoch"],
    },

    "validation": {
        "images": len(val_dataset),
        "original_mse": float(expected_mse),
        "restored_mse": float(restored_mse),
        "absolute_difference": float(absolute_difference),
        "relative_difference": float(relative_difference),
        "tolerance": float(tolerance),
        "passed": bool(absolute_difference <= tolerance),
    },

    "environment": {
        "python": platform.python_version(),
        "pytorch": torch.__version__,
    },

    "retrained": False,
}

report_path = REPORT_DIR / "restore_report.json"

with report_path.open("w", encoding="utf-8") as f:
    json.dump(report, f, indent=2)

assert report["validation"]["passed"]

print("Restore report:", report_path)
print("Independent restoration: PASSED")


Restore report: /kaggle/working/cae/hazelnut/restore-check-001/restore_report.json
Independent restoration: PASSED
